In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, EsmModel
import lightning as L
from torchmetrics.regression import PearsonCorrCoef, SpearmanCorrCoef
import scipy.stats
import sklearn.metrics as skmetrics
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

MODEL_CHECKPOINT = 'facebook/esm2_t6_8M_UR50D'
EMBED_DIM = 320  # hidden dim for t6_8M

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
esm_model = EsmModel.from_pretrained(MODEL_CHECKPOINT).to(device).eval()
print("ESM-2 (8M) loaded!")

In [ ]:
@torch.no_grad()
def get_residue_embeddings(sequence: str) -> torch.Tensor:
    """Returns per-residue embeddings (L, EMBED_DIM) — no CLS/EOS tokens."""
    inputs = tokenizer(sequence, return_tensors="pt").to(device)
    out = esm_model(**inputs)
    # strip <cls> (pos 0) and <eos> (pos -1)
    return out.last_hidden_state[0, 1:-1, :].cpu()  # (L, 320)

def find_mutation_position(wt_seq: str, mut_seq: str) -> int:
    """Returns index of first differing amino acid, or -1 if identical."""
    for i, (wt_aa, mut_aa) in enumerate(zip(wt_seq, mut_seq)):
        if wt_aa != mut_aa:
            return i
    return -1

def build_input_vector(mut_seq: str, wt_seq: str) -> torch.Tensor:
    """
    Builds a (6 * EMBED_DIM,) vector:
        [mut_at_i | wt_at_i | diff_at_i | mut_mean | wt_mean | diff_mean]
    Falls back to mean-only if no mutation position found.
    """
    mut_hidden = get_residue_embeddings(mut_seq)  # (L, 320)
    wt_hidden  = get_residue_embeddings(wt_seq)   # (L, 320)

    # global mean
    mut_mean  = mut_hidden.mean(dim=0)            # (320,)
    wt_mean   = wt_hidden.mean(dim=0)             # (320,)
    diff_mean = mut_mean - wt_mean                # (320,)

    # local at mutation position
    i = find_mutation_position(wt_seq, mut_seq)
    if i == -1:
        # no mutation found — duplicate mean as fallback
        mut_at_i  = mut_mean
        wt_at_i   = wt_mean
        diff_at_i = diff_mean
    else:
        mut_at_i  = mut_hidden[i]                 # (320,)
        wt_at_i   = wt_hidden[i]                  # (320,)
        diff_at_i = mut_at_i - wt_at_i            # (320,)

    return torch.cat([mut_at_i, wt_at_i, diff_at_i,
                      mut_mean,  wt_mean,  diff_mean])  # (1920,)


In [ ]:
class ProtSeqDataset(Dataset):
    """
    CSV must contain columns:
        mut_type  : 'wt' | anything else
        aa_seq    : mutant amino-acid sequence
        wt_seq    : wildtype amino-acid sequence
        ddG_ML    : label (only needed for non-wt rows)
    """
    def __init__(self, csv_file: str):
        df = pd.read_csv(csv_file)
        # keep only mutation rows
        df = df[df.mut_type != 'wt'].reset_index(drop=True)
        self.labels = torch.tensor(df['ddG_ML'].values, dtype=torch.float32)

        print(f"Computing embeddings for {len(df)} variants in {csv_file} ...")
        self.embeddings = []
        for idx, row in df.iterrows():
            vec = build_input_vector(row['aa_seq'], row['wt_seq'])
            self.embeddings.append(vec)
            if (idx + 1) % 100 == 0:
                print(f"  {idx + 1}/{len(df)} done")
        print("Done!")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

In [ ]:
dataset_train = ProtSeqDataset('data/mega_train.csv')
dataset_val   = ProtSeqDataset('data/mega_val.csv')

loader_train = DataLoader(dataset_train, batch_size=256, shuffle=True,  num_workers=4)
loader_val   = DataLoader(dataset_val,   batch_size=256, shuffle=False, num_workers=4)
print(f"Train: {len(dataset_train)} | Val: {len(dataset_val)}")

In [ ]:
INPUT_DIM = 6 * EMBED_DIM  # 1920

class StabModel(L.LightningModule):
    def __init__(self, input_dim=INPUT_DIM, lr=1e-4, dropout_prob=0.2):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(256, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(64, 1)
        )
        self.loss_fn      = nn.MSELoss()
        self.lr           = lr
        self.val_pearson  = PearsonCorrCoef()
        self.val_spearman = SpearmanCorrCoef()

    def forward(self, x):
        return self.model(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y  = batch
        preds = self(x)
        self.log("val_loss",  self.loss_fn(preds, y),      on_epoch=True, prog_bar=True)
        self.log("val_pear",  self.val_pearson(preds, y),  on_epoch=True, prog_bar=True)
        self.log("val_spear", self.val_spearman(preds, y), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=3, factor=0.5
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val_loss'}
        }

In [ ]:
model   = StabModel()
trainer = L.Trainer(
    devices=1,
    max_epochs=20,
    callbacks=[
        L.pytorch.callbacks.EarlyStopping(monitor='val_loss', patience=5)
    ]
)
trainer.fit(model, loader_train, loader_val)

In [ ]:
model.eval()
preds, all_y = [], []
with torch.no_grad():
    for x, y in loader_val:
        preds.append(model(x).squeeze().numpy())
        all_y.append(y.numpy())

preds = np.concatenate(preds)
all_y = np.concatenate(all_y)

print("RMSE:      ", skmetrics.mean_squared_error(all_y, preds, squared=False))
print("Pearson r: ", scipy.stats.pearsonr(preds, all_y))
print("Spearman r:", scipy.stats.spearmanr(preds, all_y))

plt.figure(figsize=(6, 6))
sns.regplot(x=preds, y=all_y, scatter_kws={'alpha': 0.3})
plt.xlabel("Predicted ddG")
plt.ylabel("Measured ddG")
plt.tight_layout()
plt.show()

In [ ]:
def predict_ddG(mut_seq: str, wt_seq: str) -> float:
    """Predict ddG for any mutant/wildtype sequence pair."""
    model.eval()
    with torch.no_grad():
        x = build_input_vector(mut_seq, wt_seq).unsqueeze(0)  # (1, 1920)
        return model(x).item()

In [ ]:
wt_sequence = str("YOUR_WT_SEQUENCE_HERE")

df_predict = pd.read_csv('data/mutated_sequences_output.csv')
print(f"Loaded {len(df_predict)} mutations")
print(df_predict.head())


In [ ]:
model.eval()

predicted_ddG = []
for idx, row in df_predict.iterrows():
    mut_seq = row['Full_Sequence']
    ddg = predict_ddG(mut_seq, wt_sequence)
    predicted_ddG.append(ddg)
    if (idx + 1) % 100 == 0:
        print(f"  {idx + 1}/{len(df_predict)} done")

df_predict['ddG_predicted'] = predicted_ddG
print("Done!")
print(df_predict.head())

In [ ]:
output_path = 'data/mutated_sequences_with_ddG.csv'
df_predict.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(df_predict[['Mutation', 'Full_Sequence', 'score', 'se', 'ddG_predicted']].head(10))

In [ ]:
df = pd.read_csv('data/mutated_sequences_with_ddG.csv')

# NOTE: ddG here is defined as dG(wt) - dG(mt)
# positive ddG = mutant less stable than WT
# flip sign below if needed:
# df['ddG_predicted'] = -df['ddG_predicted']

spearman = scipy.stats.spearmanr(df['score'], df['ddG_predicted'])
kendall  = scipy.stats.kendalltau(df['score'], df['ddG_predicted'])
pearson  = scipy.stats.pearsonr(df['score'],  df['ddG_predicted'])

print("=" * 45)
print(f"Spearman ρ: {spearman.statistic:.3f}  (p = {spearman.pvalue:.2e})")
print(f"Kendall  τ: {kendall.statistic:.3f}  (p = {kendall.pvalue:.2e})")
print(f"Pearson  r: {pearson.statistic:.3f}  (p = {pearson.pvalue:.2e})")
print("=" * 45)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter with regression line
ax = axes[0]
sns.regplot(
    data=df, x='score', y='ddG_predicted', ax=ax,
    scatter_kws={'alpha': 0.4, 's': 20},
    line_kws={'color': 'red'}
)
ax.set_xlabel("Score (from CSV)", fontsize=12)
ax.set_ylabel("Predicted ddG", fontsize=12)
ax.set_title(
    f"Score vs Predicted ddG\n"
    f"Spearman ρ = {spearman.statistic:.3f}, p = {spearman.pvalue:.2e}",
    fontsize=11
)

# Right: rank-rank scatter
ax2 = axes[1]
df['rank_score'] = df['score'].rank()
df['rank_ddG']   = df['ddG_predicted'].rank()
sns.scatterplot(
    data=df, x='rank_score', y='rank_ddG', ax=ax2,
    alpha=0.4, s=20
)
ax2.set_xlabel("Rank by Score", fontsize=12)
ax2.set_ylabel("Rank by ddG", fontsize=12)
ax2.set_title("Rank-Rank Plot", fontsize=11)

plt.tight_layout()
plt.savefig('data/score_vs_ddG_comparison.png', dpi=150)
plt.show()

# Top 10 comparisons
print("\nTop 10 by Score vs their ddG rank:")
top_score = df.nlargest(10, 'score')[['Mutation', 'score', 'ddG_predicted', 'rank_ddG']]
print(top_score.to_string(index=False))

print("\nTop 10 by ddG vs their score rank:")
top_ddg = df.nlargest(10, 'ddG_predicted')[['Mutation', 'ddG_predicted', 'score', 'rank_score']]
print(top_ddg.to_string(index=False))